## Hybrid retrieval
Semantic retrieval is performed on a vector database.

Lexical retrieval is performed using an BM25 index.

In [ ]:
# Install packages for generating embeddings and a BM25 index
%pip install -U transformers
%pip install -U chromadb
%pip install -U rank_bm25

In [ ]:
# Feature extraction functions for dense retrieval (embeddings)
import torch

def mean_pool(last_hidden_states, attention_mask):
    """
    Computes a sequence embedding by mean-pooling the embeddings of all
    non-padding tokens.

    Args:
        last_hidden_states: Tensor of shape (batch_size, seq_len, hidden_dim).
        attention_mask: Tensor of shape (batch_size, seq_len).

    Returns:
        A tensor of shape (batch_size, hidden_dim).
    """

    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)

    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def get_embedding(text, tokenizer, model):
    """
    Generates a text embedding using a transformer encoder and mean pooling.

    Args:
        text: Input text.
        tokenizer: Tokenizer associated with the model.
        model: Transformer encoder model.

    Returns:
        A tensor containing the text embedding.
    """

    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)
    embedding = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])

    return embedding

In [ ]:
# Feature extraction class for sparse retrieval (BM25)

import json
import math
import re

from rank_bm25 import BM25Okapi

class BM25Log1(BM25Okapi):
    """
    BM25 variant with strictly non-negative IDF values.

    This class extends the standard BM25Okapi implementation by replacing the
    original IDF formulation with a smoothed logarithmic variant:

        IDF(t) = log(1 + (N - df(t) + 0.5) / (df(t) + 0.5))

    This transformation guarantees:
        - IDF(t) >= 0 for all terms
        - monotonic decrease of IDF with term frequency across documents
        - elimination of negative IDF values present in classical BM25

    Rationale:
        In standard BM25, IDF can become negative when a term appears in more
        than half of the documents. While theoretically consistent with the
        probabilistic interpretation of BM25, negative IDF values may lead to
        unintuitive behavior in practical retrieval systems, where very common
        terms can penalize document scores.

        This variant removes such penalization while preserving the relative
        weighting behavior of BM25 for rare and moderately frequent terms.
    """

    def __init__(self, chunk_terms, chunk_ids, **kwargs):
        super().__init__(chunk_terms, **kwargs)

        self.chunk_ids = chunk_ids

        df = {}
        for doc in self.doc_freqs:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1

        N = self.corpus_size
        for term, dfi in df.items():
            self.idf[term] = math.log(1 + (N - dfi + 0.5) / (dfi + 0.5))

    @staticmethod
    def extract_terms(text):
        """
        Extracts lowercase alphabetic terms from a text for BM25 indexing
        and keyword-based retrieval.

        Args:
            text: Input text.

        Returns:
            A list of normalized terms.
        """

        return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())


    def save(self, path):
        """Serialize BM25 index to JSON"""
        data = {
            "chunk_ids": self.chunk_ids,
            "idf": {k: float(v) for k, v in self.idf.items()},
            "doc_freqs": self.doc_freqs,
            "doc_len": self.doc_len,
            "avgdl": self.avgdl,
            "corpus_size": self.corpus_size,
            "k1": self.k1,
            "b": self.b
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path):
        """Load BM25 index from JSON without recomputing statistics."""
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        bm25 = cls.__new__(cls)
        bm25.chunk_ids = data["chunk_ids"]
        bm25.idf = {k: float(v) for k, v in data["idf"].items()}
        bm25.doc_freqs = data["doc_freqs"]
        bm25.doc_len = data["doc_len"]
        bm25.avgdl = data["avgdl"]
        bm25.corpus_size = data["corpus_size"]
        bm25.k1 = data["k1"]
        bm25.b = data["b"]

        return bm25

In [ ]:
# Functions for Retrieval
import torch

def rrf(ranks, k=60):
    """
    Compute Reciprocal Rank Fusion (RRF) score.

    Each rank contributes 1 / (rank + k), where k smooths high ranks.
    Lower ranks contribute more to the final score.

    Args:
        ranks (iterable of int/float): Positive ranks (1-based).
        k (int, optional): Smoothing constant. Default is 60.

    Returns:
        float: RRF score.

    Raises:
        ValueError: If ranks is empty or contains non-positive values.
    """

    if not ranks:
        raise ValueError("At least one rank is required.")

    score = 0.0
    for rank in ranks:
        if rank <= 0:
            raise ValueError("All ranks must be greater than 0.")
        score += 1 / (rank + k)
    return score


def semantic_search(query_text, collection, tokenizer, model, top_k=50):
    """
    Performs semantic (dense vector) search over a ChromaDB collection using transformer embeddings.

    This function encodes the input query into a dense vector representation using a transformer
    model, queries a vector database (ChromaDB) using cosine similarity, and returns the top-k
    most similar chunks ranked by semantic relevance.

    Args:
        query_text (str):
            Input query string to be embedded and searched.

        collection:
            ChromaDB collection containing precomputed chunk embeddings and metadata.

        tokenizer:
            Tokenizer associated with the transformer model used for embedding generation.

        model:
            Transformer encoder model used to compute query embeddings.

       top_k (int, optional):
            Number of top-ranked results to return. Defaults to 50.

    Returns:
       List[Tuple[str, float]]:
          A list of (chunk_id, similarity) pairs sorted in descending order of semantic relevance.

    Notes:
        - Higher similarity values indicate higher semantic relevance.
        - Similarity is computed as: 1 - cosine_distance returned by ChromaDB.
        - This function performs ranking only; it does not retrieve full chunk text.
        - Retrieval of full documents must be performed separately using chunk_id.
    """
    query_embedding = get_embedding(query_text, tokenizer, model).squeeze().tolist()
    chroma_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["distances"]
        )
    return [(chunk_id, 1 - distance)
            for chunk_id, distance in zip(
                chroma_results["ids"][0],
                chroma_results["distances"][0]
        )
    ]


def lexical_search(query_text, bm25, top_k=50):
    """
    Performs lexical (sparse) search over a BM25 index and returns ranked chunk identifiers.

    This function tokenizes the input query, computes BM25 relevance scores over all indexed
    chunks, and returns the top-k chunks ordered by decreasing lexical relevance.

    Args:
        query_text (str):
            Input query string to be tokenized and searched.

        bm25 (BM25Log1):
            BM25 index object containing precomputed document frequencies, IDF values,
            and chunk identifier mappings.

        top_k (int, optional):
            Number of top-ranked results to return. Defaults to 50.

    Returns:
        List[Tuple[str, float]]:
            A list of (chunk_id, BM25_score) pairs sorted in descending order of
            lexical relevance score.

    Notes:
        - Higher BM25 scores indicate higher lexical relevance.
        - This function performs ranking only; it does not retrieve chunk text or metadata.
        - Retrieval of full documents must be performed separately using chunk_id.
    """

    query_terms = BM25Log1.extract_terms(query_text)
    print(query_terms)
    bm25_scores = bm25.get_scores(query_terms)
    sorted_idx = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:top_k]

    return [(bm25.chunk_ids[i], float(bm25_scores[i])) for i in sorted_idx]


def hybrid_ranking(bm25_results, semantic_results, top_n=20):
    """
    Fuses lexical (BM25) and semantic (dense vector) ranking lists using Reciprocal Rank Fusion (RRF).

    This function does not perform any search itself. It takes precomputed ranked results
    from a BM25 lexical search and a semantic vector search, converts them into rank-based
    representations, and combines them into a single unified ranking using RRF.

    Args:
        bm25_results (List[Tuple[str, float]]):
            Ranked BM25 results as (chunk_id, score), ordered by decreasing lexical relevance.

        semantic_results (List[Tuple[str, float]]):
            Ranked semantic results as (chunk_id, distance), ordered by increasing
            cosine distance (most similar first).

        top_n (int, optional):
            Number of top fused results to return. Defaults to 20.

    Returns:
        List[Tuple[str, float]]:
            A list of (chunk_id, rrf_score) pairs sorted in descending order of fused relevance.

    Notes:
        - This function performs rank fusion only; it does not compute BM25 scores or embeddings.
        - RRF uses rank positions, not raw scores or distances.
        - Documents missing from one ranking are still included via union fusion.
    """

    bm25_rank = {
        chunk_id: rank + 1
        for rank, (chunk_id, _) in enumerate(bm25_results)
    }

    semantic_rank = {
        chunk_id: rank + 1
        for rank, (chunk_id, _) in enumerate(semantic_results)
    }

    all_chunk_ids = set(bm25_rank) | set(semantic_rank)

    ranked = []

    for chunk_id in all_chunk_ids:
        ranks = []

        b_rank = bm25_rank.get(chunk_id)
        if b_rank is not None:
            ranks.append(b_rank)

        s_rank = semantic_rank.get(chunk_id)
        if s_rank is not None:
            ranks.append(s_rank)

        ranked.append(
            (
                chunk_id,
                rrf(ranks),
                b_rank,
                s_rank
            )
        )

    ranked.sort(key=lambda x: x[1], reverse=True)

    return ranked[:top_n]

In [ ]:
import chromadb
from google.colab import drive
from transformers import AutoTokenizer, AutoModel

# Mount Google Drive for file access
drive.mount("/content/drive", force_remount=False)

# Model used for embedding generation: Multilingual E5 Base
model_name = "intfloat/multilingual-e5-base"

# Load the model and tokenizer from Hugging Face Hub
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

DB_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
db_file_name = "chroma_db_1781251016" # @param {type:"string"}
db_file_path = f"{DB_DIR}/{db_file_name}"
collection_name = "construction_site_visit_reports"

BM25_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
bm25_file_name = "bm25_1781251016.json" # @param {type:"string"}
bm25_file_path = f"{BM25_DIR}/{bm25_file_name}"

# Open the vector database
client = chromadb.PersistentClient(path=db_file_path)

# Get the collection
print(f"{collection_name = }")
collection = client.get_collection(collection_name)

# Load the BM25 index
bm25 = BM25Log1.load(bm25_file_path)


In [ ]:
print(db_file_path)
for c in client.list_collections():
    print(c.name)

In [ ]:
# Query text
query_text = "¿Qué problemas pueden afectar a la seguridad de la explotación ferroviaria?"  # @param {type:"string"}

# Search and ranking
semantic_results = semantic_search(query_text, collection, tokenizer, model, top_k=50)
bm25_results = lexical_search(query_text, bm25, top_k=50)
hybrid_results = hybrid_ranking(bm25_results, semantic_results, top_n=5)

bm25_dict = dict(bm25_results)
semantic_dict = dict(semantic_results)

hybrid_map = {
    cid: {
        "rrf_score": rrf,
        "bm25_rank": bm25_r,
        "semantic_rank": sem_r
    }
    for cid, rrf, bm25_r, sem_r in hybrid_results
}

final_chunk_ids = [
    cid for cid, _, _, _ in hybrid_results
]

# Retrieval
retrieved = collection.get(
    ids=final_chunk_ids,
    include=["documents", "metadatas"]
)
id_to_doc = dict(zip(retrieved["ids"], retrieved["documents"]))
id_to_meta = dict(zip(retrieved["ids"], retrieved["metadatas"]))


# Build final output
final_chunks = [
    {
        "chunk_id": cid,
        "text": id_to_doc[cid],
        "metadata": id_to_meta[cid],

        # BM25 score
        "bm25_score": bm25_dict.get(cid),

        # semantic score (en tu caso es distancia coseno)
        "cosine_distance": semantic_dict.get(cid),

        # hybrid (RRF + ranks)
        "rrf_score": hybrid_map[cid]["rrf_score"],
        "bm25_rank": hybrid_map[cid]["bm25_rank"],
        "semantic_rank": hybrid_map[cid]["semantic_rank"]
    }
    for cid in final_chunk_ids
]

In [ ]:
for chunk in final_chunks:
  print(f"chunk_id: {chunk["chunk_id"]}")
  print(f"text: {chunk["text"]}")
  print(f"cosine_distance: {chunk["cosine_distance"]}")
  print(f"bm25_score: {chunk["bm25_score"]}")
  print(f"semantic_rank: {chunk["semantic_rank"]}")
  print(f"bm25_rank: {chunk["bm25_rank"]}")
  print(f"rrf_score: {chunk["rrf_score"]}")
